In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_tavily import TavilySearch
from langchain_core.tools import tool

import requests
import random
import os

In [4]:
load_dotenv()

True

In [5]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.7,
)

In [7]:
search_tool = TavilySearch(
    max_results=3,
    api_key=os.getenv("TAVILY_API_KEY"),
    engine_id=os.getenv("TAVILY_ENGINE_ID"),
    topic="general",
    search_type="web",
    search_depth="advanced"
)

In [8]:
@tool
def calculator(first_num: float, seconf_num: float, operation: str) -> dict:
    """A simple calculator tool that performs basic arithmetic operations."""
    
    try:
        if operation == "add":
            result = first_num + seconf_num
        elif operation == "subtract":
            result = first_num - seconf_num
        elif operation == "multiply":
            result = first_num * seconf_num
        elif operation == "divide":
            if seconf_num == 0:
                return {"error": "Division by zero is not allowed."}
            result = first_num / seconf_num
        else:
            return {"error": f"Invalid operation: {operation}. Supported operations are add, subtract, multiply, divide."}
        
        return {"result": result}
    except Exception as e:
        return {"error": str(e)}
      

In [9]:
# pip install yfinance
import yfinance as yf

@tool
def get_stock_price(ticker: str) -> dict:
    """Fetches the current stock price for a given ticker symbol."""
    try:
        stock = yf.Ticker(ticker)
        # Get the most recent price
        price = stock.history(period="1d")['Close'].iloc[-1]
        return {"ticker": ticker.upper(), "price": float(price), "currency": "USD"}
    except Exception as e:
        return {"error": str(e)}   

In [10]:
tools = [calculator, get_stock_price, search_tool]

llm_with_tools = model.bind_tools(tools)

In [11]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [12]:
def chat_node(state: ChatState): 
    """LLM node that may answer a question or call a tool based on the input messages."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}
  
tool_node = ToolNode(tools)

In [13]:
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

In [14]:
graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)             

In [15]:
chatbot = graph.compile()

In [17]:
out = chatbot.invoke({"messages": [HumanMessage(content="What is the current stock price of AAPL?")]})

print(out["messages"])

[HumanMessage(content='What is the current stock price of AAPL?', additional_kwargs={}, response_metadata={}, id='b902da6e-e7b8-4efc-bb7d-28e96e5f07ff'), AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fac9c-a8fd-7641-b37a-c1c3aefd6b2a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1567, 'output_tokens': 0, 'total_tokens': 1567, 'input_token_details': {'cache_read': 0}})]
